# Tugas Mandiri Pertemuan 10: Dashboard Pesanan Real Time Kopi Nusantara

**Nama:** Hilmi Mufid | **NPM:** 2505060046 | **Program Studi:** S1 Teknologi Informasi

**Skenario:** Manajemen Kopi Nusantara ingin memantau performa tiap cabang setiap saat, bukan hanya lewat laporan bulanan. Notebook ini membangun purwarupa pipeline streaming dengan Spark Structured Streaming, dengan aliran data yang disimulasikan lewat folder.

Seluruh folder streaming dibuat di folder kerja notebook (disk lokal), sehingga Hadoop tidak perlu menyala.

Membuat `SparkSession` (jumlah partisi shuffle dikurangi menjadi 4 agar pemrosesan data kecil lebih cepat):

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import col, sum as spark_sum, count, window
import os
import shutil
import time
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas10-StreamingKopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


## A. Menyiapkan Streaming Source

Skema pesanan didefinisikan secara eksplisit dengan `StructType`, folder simulasi `tugas10_stream` dibuat kosong, lalu `readStream` didefinisikan dari folder tersebut.

In [5]:
# Membersihkan sisa folder percobaan sebelumnya agar notebook aman dijalankan ulang (Run All)
folder_tugas = "tugas10_stream"
for folder in [folder_tugas, "tugas10_output", "tugas10_checkpoint"]:
    shutil.rmtree(folder, ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

# Skema eksplisit data pesanan (wajib untuk streaming)
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])

# Streaming source: folder tugas10_stream (masih kosong)
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas)

print("Folder streaming siap:", folder_tugas)
print("df_stream.isStreaming:", df_stream.isStreaming)
df_stream.printSchema()

Folder streaming siap: tugas10_stream
df_stream.isStreaming: True
root
 |-- pesanan_id: string (nullable = true)
 |-- nama_cabang: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- waktu_pesan: timestamp (nullable = true)



## B. Agregasi per Cabang

Kolom turunan `total_harga` (`qty x harga_satuan`) ditambahkan, lalu `groupBy("nama_cabang")` menghitung total pendapatan dan jumlah pesanan. Query berjalan dengan sink `memory` bernama `agregasi_cabang`.

In [6]:
# Kolom turunan dibuat pada DataFrame baru, sehingga df_stream tetap berisi data mentah (dipakai lagi di bagian E)
df_stream_total = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_cabang = df_stream_total.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query1 = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Query 1 (agregasi per cabang) dimulai. Aktif:", query1.isActive)

Query 1 (agregasi per cabang) dimulai. Aktif: True


## C. Agregasi Berbasis Window Waktu

Streaming query kedua membaca folder yang sama, dikelompokkan per window waktu 2 menit dari kolom `waktu_pesan`, dengan sink `memory` bernama `agregasi_waktu`.

In [7]:
# readStream didefinisikan ulang dari folder yang sama, satu sumber boleh dibaca banyak query
df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas) \
    .withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_waktu = df_stream_waktu.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query2 = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Query 2 (window waktu 2 menit) dimulai. Aktif:", query2.isActive)
print("Jumlah streaming query yang sedang berjalan:", len(spark.streams.active))

Query 2 (window waktu 2 menit) dimulai. Aktif: True
Jumlah streaming query yang sedang berjalan: 2


## D. Menguji dengan Data Simulasi

Empat batch data pesanan dikirim bergantian (masing masing satu berkas CSV baru, dengan jeda `time.sleep()` agar Spark sempat memproses). Setelah setiap batch, hasil `agregasi_cabang` ditampilkan untuk membuktikan angka totalnya terus terakumulasi. Batch 4 dikirim pada bagian E.

In [10]:
kolom_pesanan = ["pesanan_id", "nama_cabang", "nama_menu", "qty", "harga_satuan", "waktu_pesan"]

def kirim_batch(nomor, baris, jeda=10):
    # Menulis satu berkas CSV baru ke folder yang diawasi Spark, lalu menunggu micro batch diproses
    pd.DataFrame(baris, columns=kolom_pesanan).to_csv(f"{folder_tugas}/batch_{nomor}.csv", index=False)
    print(f"Batch {nomor} dikirim ({len(baris)} pesanan). Menunggu Spark memproses...")
    time.sleep(jeda)

**Batch 1**

In [11]:
kirim_batch(1, [
    ["P1", "Kopi Nusantara Magelang", "Espresso", 2, 18000, "2026-11-05 10:00:30"],
    ["P2", "Kopi Nusantara Yogyakarta", "Latte", 1, 27000, "2026-11-05 10:01:10"],
    ["P3", "Kopi Nusantara Magelang", "Roti Bakar", 3, 15000, "2026-11-05 10:01:45"],
])

spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show(truncate=False)

Batch 1 dikirim (3 pesanan). Menunggu Spark memproses...
+-------------------------+----------------+--------------+
|nama_cabang              |total_pendapatan|jumlah_pesanan|
+-------------------------+----------------+--------------+
|Kopi Nusantara Magelang  |81000           |2             |
|Kopi Nusantara Yogyakarta|27000           |1             |
+-------------------------+----------------+--------------+



**Batch 2**

In [12]:
kirim_batch(2, [
    ["P4", "Kopi Nusantara Semarang", "Cappuccino", 2, 25000, "2026-11-05 10:02:20"],
    ["P5", "Kopi Nusantara Yogyakarta", "Kopi Susu Gula Aren", 4, 22000, "2026-11-05 10:03:05"],
    ["P6", "Kopi Nusantara Magelang", "Matcha Latte", 1, 28000, "2026-11-05 10:03:40"],
])

spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show(truncate=False)

Batch 2 dikirim (3 pesanan). Menunggu Spark memproses...
+-------------------------+----------------+--------------+
|nama_cabang              |total_pendapatan|jumlah_pesanan|
+-------------------------+----------------+--------------+
|Kopi Nusantara Yogyakarta|115000          |2             |
|Kopi Nusantara Magelang  |109000          |3             |
|Kopi Nusantara Semarang  |50000           |1             |
+-------------------------+----------------+--------------+



**Batch 3**

In [13]:
kirim_batch(3, [
    ["P7", "Kopi Nusantara Semarang", "Americano", 3, 20000, "2026-11-05 10:04:15"],
    ["P8", "Kopi Nusantara Semarang", "Kentang Goreng", 2, 20000, "2026-11-05 10:04:50"],
    ["P9", "Kopi Nusantara Yogyakarta", "Es Teh Lemon", 5, 12000, "2026-11-05 10:05:10"],
    ["P10", "Kopi Nusantara Magelang", "Cokelat Panas", 2, 24000, "2026-11-05 10:05:30"],
])

spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show(truncate=False)

Batch 3 dikirim (4 pesanan). Menunggu Spark memproses...
+-------------------------+----------------+--------------+
|nama_cabang              |total_pendapatan|jumlah_pesanan|
+-------------------------+----------------+--------------+
|Kopi Nusantara Yogyakarta|175000          |3             |
|Kopi Nusantara Magelang  |157000          |4             |
|Kopi Nusantara Semarang  |150000          |3             |
+-------------------------+----------------+--------------+



Hasil `agregasi_waktu` setelah tiga batch:

In [14]:
# Hasil query 2: total per window waktu 2 menit dari ketiga batch
spark.sql("""
    SELECT window.start, window.end, total_pendapatan, jumlah_pesanan
    FROM agregasi_waktu
    ORDER BY window.start
""").show(truncate=False)

+-------------------+-------------------+----------------+--------------+
|start              |end                |total_pendapatan|jumlah_pesanan|
+-------------------+-------------------+----------------+--------------+
|2026-11-05 10:00:00|2026-11-05 10:02:00|108000          |3             |
|2026-11-05 10:02:00|2026-11-05 10:04:00|166000          |3             |
|2026-11-05 10:04:00|2026-11-05 10:06:00|208000          |4             |
+-------------------+-------------------+----------------+--------------+



## E. Sink Berbasis Berkas (Tantangan)

Streaming query ketiga membaca `df_stream` (data mentah, tanpa agregasi) dan menuliskannya ke folder `tugas10_output` dalam format Parquet. Setelah itu satu batch baru dikirim, isi folder diperiksa, dan berkas Parquet dibaca kembali sebagai DataFrame batch biasa.

In [16]:
# Query 3: menulis data MENTAH (tanpa agregasi) ke berkas Parquet secara berkelanjutan
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("Query 3 (file sink Parquet) dimulai. Aktif:", query3.isActive)
print("Jumlah streaming query yang sedang berjalan:", len(spark.streams.active))

Query 3 (file sink Parquet) dimulai. Aktif: True
Jumlah streaming query yang sedang berjalan: 3


In [17]:
kirim_batch(4, [
    ["P11", "Kopi Nusantara Semarang", "Latte", 2, 27000, "2026-11-05 10:06:20"],
    ["P12", "Kopi Nusantara Magelang", "Americano", 1, 20000, "2026-11-05 10:06:45"],
], jeda=12)

print("Isi folder tugas10_output:")
!ls tugas10_output

Batch 4 dikirim (2 pesanan). Menunggu Spark memproses...
Isi folder tugas10_output:
_spark_metadata
part-00000-72e63a0f-6ed7-4d50-b3ef-0278b6277c27-c000.snappy.parquet
part-00000-7ea0d76b-a2ea-4349-8f3b-6963ec7bf14d-c000.snappy.parquet
part-00001-0c64105d-8411-4f05-a882-d7bdf89c0b01-c000.snappy.parquet
part-00002-5fefc0db-c212-4815-83b5-55f90054e5f8-c000.snappy.parquet


In [18]:
# Membaca kembali berkas Parquet sebagai DataFrame batch biasa
df_hasil = spark.read.parquet("tugas10_output")
print("Jumlah baris di berkas Parquet:", df_hasil.count())
df_hasil.orderBy("waktu_pesan").show(12, truncate=False)

# Dua query lainnya ikut memproses batch 4 dari folder yang sama
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show(truncate=False)

Jumlah baris di berkas Parquet: 12
+----------+-------------------------+-------------------+---+------------+-------------------+
|pesanan_id|nama_cabang              |nama_menu          |qty|harga_satuan|waktu_pesan        |
+----------+-------------------------+-------------------+---+------------+-------------------+
|P1        |Kopi Nusantara Magelang  |Espresso           |2  |18000       |2026-11-05 10:00:30|
|P2        |Kopi Nusantara Yogyakarta|Latte              |1  |27000       |2026-11-05 10:01:10|
|P3        |Kopi Nusantara Magelang  |Roti Bakar         |3  |15000       |2026-11-05 10:01:45|
|P4        |Kopi Nusantara Semarang  |Cappuccino         |2  |25000       |2026-11-05 10:02:20|
|P5        |Kopi Nusantara Yogyakarta|Kopi Susu Gula Aren|4  |22000       |2026-11-05 10:03:05|
|P6        |Kopi Nusantara Magelang  |Matcha Latte       |1  |28000       |2026-11-05 10:03:40|
|P7        |Kopi Nusantara Semarang  |Americano          |3  |20000       |2026-11-05 10:04:15|
|P8  

Menghentikan seluruh streaming query dan `SparkSession`:

In [19]:
query1.stop()
query2.stop()
query3.stop()
print("Status aktif query 1, 2, 3:", query1.isActive, query2.isActive, query3.isActive)
print("Jumlah streaming query yang masih berjalan:", len(spark.streams.active))

spark.stop()
print("SparkSession ditutup.")

Status aktif query 1, 2, 3: False False False
Jumlah streaming query yang masih berjalan: 0
SparkSession ditutup.
